In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df =spark.read.format("csv")\
    .option("header","true")\
    .option("inferSchema","true")\
        .load("/Volumes/pyspark/source/ff_source/BigMart Sales.csv")

In [0]:
df.display()

####SPLIT and Indexing

SPLIT: It is used to split the data and it will store data in list


#####Split

In [0]:
df.withColumn('Outlet_Type',split(col('Outlet_Type'),' ')).display()

#####Indexing

In [0]:
df.withColumn('Outlet_Type',split(col('Outlet_Type'),' ')[1]).display()

####EXPLODE

If your column is in array and you explode the values into new rows then use explode or its like 1nf in SQL (list will split into single values and it creates a new row)

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))

df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

####ARRAY_CONTAINS

In a Array if the particular value which you want if it is present then it will return yes else no

In [0]:
df_exp.display()

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

####GROUP_BY

It is used to group the data based on a common value in a column and mainly it is used when you are using aggregation with them

In [0]:
df.display()

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type").agg(max("Item_MRP")).display()

In [0]:
df.dropDuplicates(['Item_Type']).sort('Item_MRP').display()

In [0]:
item_type_sum = df.groupBy("Item_Type").agg(sum("Item_MRP").alias("sum_Item_MRP"))
item_type_sum.agg(max("sum_Item_MRP")).display()

In [0]:
x = df.groupBy("Item_Type").agg(count("Item_MRP").alias("count_Item_MRP"))
x.agg(sum("count_Item_MRP")).display()

In [0]:
df.groupBy("Item_Type").agg(round(avg("Item_MRP"),1).alias("avg_Item_MRP")).display()

In [0]:
df.display()

In [0]:
y = df.groupBy("Item_Type","Outlet_Size").agg(count("Item_MRP").alias("ccount")).display()

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(round(sum("Item_MRP"),1).alias("sum_Item_MRP")).sort(['Item_Type']).fillna('not available').display()

In [0]:
x = df.groupBy("Item_Type","Outlet_Size").agg(round(sum("Item_MRP"),1).alias("sum_Item_MRP"))
x.filter(col("sum_Item_MRP") == 59047.2).display()


In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP").alias("Total_mrp"),avg("Item_MRP").alias("avg_Item_MRP")).display()

####COLLECT_LIST
It is used to collect the group of values and displays as a list

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(collect_list("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type","Outlet_Size")\
    .agg(expr("transform(collect_list(Item_MRP), x -> round(x)) as list_in_int")).display()

####PIVOT
It is used to transform rows into columns and it is used with group by

In [0]:
df.groupBy("Item_Type")\
    .pivot("Outlet_Size")\
        .agg(count("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type","Outlet_Size")\
    .agg(count("Item_MRP")).display()

####WHEN-OTHERWISE
It is like case when function in SQL

In [0]:
df.withColumn('item_types',when((col('Item_Type') == 'Dairy') | (col('Item_Type') =='Soft Drink'), 'drinks').otherwise('not drinks')).display()

In [0]:
df.withColumn('type',when(col("Item_Type") == 'Meat' ,'Non-Veg').otherwise('Veg')).filter(col("type") == 'Non-Veg').display()

In [0]:
df.withColumn('veg-flag',when(col("Item_Type") == 'Meat' ,'Non-Veg').otherwise(when(col("Item_MRP") >= 100, 'veg-expensive').otherwise('veg-cheap'))).display()

####JOINS
It is used to join one or more dfs in pyspark (similar to joins in SQL)

In [0]:
data1 = [('1', 'gurav','d01'),
         ('2', 'suresh','d02'),
         ('3', 'ramesh','d03'),
         ('4', 'rakesh','d13'),
         ('5', 'rajesh','d05'),
         ('6', 'mahesh','d06')]
df1 = spark.createDataFrame(data = data1, schema = ['id', 'name','deptid'])
df1.display()

In [0]:
data2 = [('d01', 'hr'),
         ('d02', 'it'),
         ('d03', 'finance'),
         ('d04', 'marketing'),
         ('d05', 'csm'),
         ('d06', 'r and d'),
         ('d07', 'sales')]
df2 = spark.createDataFrame(data = data2, schema = ['deptid', 'dept_name'])
df2.display()

####INNER_JOIN

In [0]:
df1.join(df2, df1.deptid == df2.deptid,'inner').display()

In [0]:
df1.join(df2, df1.deptid == df2.deptid,'inner').select(df1['id'], df1['name'], df1['deptid'], df2['dept_name']).display()

####LEFT_JOIN

In [0]:
df1.join(df2, df1['deptid'] == df2['deptid'],'left').display()


In [0]:
df1.join(df2, df1['deptid'] == df2['deptid'],'left').select(df1['deptid'], df1['name'], df2['dept_name']).display()


####RIGHT_JOIN

In [0]:
df1.join(df2, df1['deptid'] == df2['deptid'],'right').display()

####FULL_OUTER_JOIN

In [0]:
df1.join(df2, df1['deptid'] == df2['deptid'],'outer').display()

####ANTI OR LEFT_ANTI

It is used to fetch a data from one df (df1) which is not present in another df (df2)

In [0]:
df1.join(df2, df1['deptid'] == df2['deptid'],"anti").display()

####SEMI OR LEFT_SEMI

It is used to fetch data from onde df (df1) which is present in another df (df2)

In [0]:
df1.join(df2, df1['deptid'] == df2['deptid'],"leftsemi").display()

####DATA_WRITING

In [0]:
df.display()

Writing data in csv file

In [0]:
trans_df = df.select("Item_Identifier","Item_MRP","Item_Type")
trans_df.display()

In [0]:
# trans_df.write.format("csv")\
#     .option("header", "true")\
#     .save("/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv")

In [0]:
x= spark.read.format("parquet")\
    .option("header", "true")\
    .load("/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv")
x.display()

In [0]:
trans_df2 = df.filter(col("Item_Type") == 'Dairy').select("Item_Identifier","Item_MRP","Item_Type")
trans_df2.display()

In [0]:
trans_df2 = trans_df2.withColumn("Item_Type",when(col("Item_Type") == 'Dairy',"Milk_Products").otherwise(col("Item_Type")))

In [0]:
trans_df2.display()

Append mode in data writing

In [0]:
trans_df2.write.format("csv")\
    .mode("append")\
        .option('path','/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv')\
            .option("header", "true")\
             .save()

In [0]:
x.display()

Overwrite mode in data writing

In [0]:
trans_df2 = trans_df2.withColumn("Item_Type",when(col("Item_Type") == 'Milk_Products',"Dairy").otherwise(col("Item_Type")))
trans_df2.write.format("csv")\
    .mode("overwrite")\
        .option('path','/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv')\
            .option("header", "true")\
             .save()

Error mode in data writing

In [0]:
# trans_df.write.format("csv")\
#     .mode("error")\
#         .option('path','/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv')\
#             .option("header", "true")\
#              .save()

Ignore mode in data writing

In [0]:
trans_df2.write.format("csv")\
    .mode("ignore")\
        .option('path','/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv')\
            .option("header", "true")\
                .save()

####PARQUET FILE FORMAT

In [0]:
trans_df2.write.format("parquet")\
    .mode("overwrite")\
        .option('path','/Volumes/pyspark/source/ff_source/Trans_BigMart Sales.csv')\
            .option("header", "true")\
                .save()

####TABLE

writing data in table

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .saveAsTable('my_tabel')

####SPARK_SQL

In [0]:
df.display()

Whenever you want to use SQL on top of your df just convert df into temp_view (object)

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view where Item_Type == 'Dairy'

In [0]:
df_sql = spark.sql("select * from my_view where Item_Type == 'Dairy'")

In [0]:
df_sql.display()